# Databricks setup: Supply Chain Genie vs Cortex eval
Run top to bottom. The **UI** steps (metric view, Genie Agent) are marked **[UI]**; do them when you reach them.
Set your own names in section 0. The data is in the repo under `data/output/` (same files for everyone).

In [ ]:
%pip install -q databricks-ai-search
dbutils.library.restartPython()

## 0. Configuration (edit these values)

In [ ]:
CATALOG     = "<your_catalog>"       # existing catalog you can create objects in
SCHEMA      = "supply_chain_eval"    # created if missing
VOLUME_NAME = "landing"              # managed volume, created if missing (or point at an existing external volume)
VS_ENDPOINT = "sc-eval-vs"           # Vector Search endpoint, created if missing (any name)

FQ     = f"{CATALOG}.{SCHEMA}"
VOLUME = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME_NAME}"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {FQ}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {FQ}.{VOLUME_NAME}")
spark.sql(f"USE CATALOG {CATALOG}")
spark.sql(f"USE SCHEMA {SCHEMA}")
print(FQ, VOLUME)

## 1. Raw tables
Upload `data/output/csv/*.csv` → `<VOLUME>/csv/` and `data/output/contracts/*.pdf` → `<VOLUME>/contracts/` (Catalog Explorer → `<SCHEMA>.<VOLUME_NAME>` → **Upload to this volume**), then run the cells below.

In [ ]:
# Confirm the uploaded files are visible
for sub in ["csv", "contracts"]:
    files = [f.name for f in dbutils.fs.ls(f"{VOLUME}/{sub}")]
    print(sub, len(files), files)   # expect csv: 6 files, contracts: 14 PDFs

In [ ]:
SCHEMAS = {
    "dim_supplier": "supplier_id STRING, supplier_name STRING, supplier_region STRING, tier STRING, contract_id STRING, contract_grace_days INT",
    "dim_plant": "plant_id STRING, plant_name STRING, plant_region STRING, country STRING",
    "dim_component": "component_id STRING, component_name STRING, category STRING, unit_cost DECIMAL(12,2), criticality STRING",
    "fact_purchase_orders": "po_id STRING, supplier_id STRING, plant_id STRING, component_id STRING, order_date DATE, promised_date DATE, actual_delivery_date DATE, quantity INT, unit_price DECIMAL(12,2), status STRING, is_late BOOLEAN",
    "fact_inventory_daily": "snapshot_date DATE, plant_id STRING, component_id STRING, on_hand_qty INT, safety_stock_qty INT",
    "fact_downtime_events": "event_id STRING, plant_id STRING, component_id STRING, supplier_id STRING, event_date DATE, event_type STRING, downtime_hours DECIMAL(6,1), cost_per_hour DECIMAL(10,2)",
}
for t, sch in SCHEMAS.items():
    (spark.read.option("header", True).option("nullValue", "").schema(sch).csv(f"{VOLUME}/csv/{t}.csv")
         .write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{FQ}.{t}"))
# expect 12 / 5 / 40 / 3000 / 18250 / 150
display(spark.sql(" UNION ALL ".join(f"SELECT '{t}' tbl, COUNT(*) n FROM {FQ}.{t}" for t in SCHEMAS)))

## 2. [UI] Metric view generated by Genie
Catalog Explorer → `<CATALOG>.<SCHEMA>` → **Create → Metric view** → **Genie Code (Agent mode)**, paste:
```
Create a metric view named mv_supply_chain in <CATALOG>.<SCHEMA> over the tables
dim_supplier, dim_plant, dim_component, fact_purchase_orders, fact_inventory_daily, fact_downtime_events.
Supply chain performance for Contoso Manufacturing in 2025: suppliers, plants, components, purchase orders, daily inventory and downtime events.
```
Save **without edits** (write down any fix you had to make). Review the generated metric view in Catalog Explorer.

## 3. Contracts: parse → chunk → Vector Search index → UC search function

In [ ]:
spark.sql(f"""
CREATE OR REPLACE TABLE {FQ}.contract_docs AS
SELECT regexp_extract(path, '([^/]+)\\.pdf$', 1) AS doc_name,
       regexp_extract(path, '(SUP-[0-9]{{3}})', 1) AS supplier_id,
       CASE WHEN path ILIKE '%AMENDMENT%' THEN 'AMENDMENT' ELSE 'MSA' END AS doc_type, path,
       concat_ws('\n', transform(try_cast(ai_parse_document(content):document:elements AS ARRAY<VARIANT>),
                                 e -> try_cast(e:content AS STRING))) AS doc_text
FROM READ_FILES('{VOLUME}/contracts', format => 'binaryFile')""")

from pyspark.sql import functions as F, types as T
def chunk(text, size=1200, overlap=150):
    out, cur = [], ""
    for p in (text or "").split("\n"):
        if cur and len(cur) + len(p) > size:
            out.append(cur); cur = cur[-overlap:]
        cur = f"{cur}\n{p}" if cur else p
    return out + ([cur] if cur else [])
# Every chunk starts with a document header, so later chunks (e.g. section 8) stay findable by supplier
sup = spark.table(f"{FQ}.dim_supplier").select("supplier_id", "supplier_name")
(spark.table(f"{FQ}.contract_docs").join(sup, "supplier_id", "left")
   .select("doc_name", "supplier_id", "supplier_name", "doc_type",
           F.posexplode(F.udf(chunk, T.ArrayType(T.StringType()))("doc_text")).alias("chunk_index", "chunk"))
   .withColumn("chunk", F.concat(F.lit("["), F.col("doc_name"), F.lit(" | "), F.col("supplier_id"), F.lit(" | "),
                                 F.col("supplier_name"), F.lit("]\n"), F.col("chunk")))
   .withColumn("chunk_id", F.concat_ws("-", "doc_name", "chunk_index"))
   .drop("supplier_name")
   .write.mode("overwrite").option("overwriteSchema", True).option("delta.enableChangeDataFeed", "true")
   .saveAsTable(f"{FQ}.contract_chunks"))
display(spark.sql(f"SELECT doc_name, COUNT(*) chunks, min(left(chunk, 60)) header FROM {FQ}.contract_chunks GROUP BY 1 ORDER BY 1"))  # expect 14 docs

In [ ]:
VS_INDEX = f"{FQ}.contract_chunks_idx"   # uses FQ and VS_ENDPOINT from section 0 (re-run it after a kernel restart)

# Embedding model = a model serving endpoint; use the first one that exists in this workspace
from databricks.sdk import WorkspaceClient
_available = {e.name for e in WorkspaceClient().serving_endpoints.list()}
EMBEDDING_MODEL = next((m for m in ["databricks-gte-large-en", "databricks-bge-large-en"] if m in _available), None)
assert EMBEDDING_MODEL, "No embedding endpoint found - check Serving and set EMBEDDING_MODEL manually"
print("Embedding model:", EMBEDDING_MODEL)

try:
    from databricks.ai_search.client import VectorSearchClient      # new package name (databricks-ai-search)
except ImportError:
    from databricks.vector_search.client import VectorSearchClient  # older runtimes
vsc = VectorSearchClient(disable_notice=True)                       # notebook token auth is fine for this eval
if VS_ENDPOINT not in [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]:
    vsc.create_endpoint_and_wait(name=VS_ENDPOINT, endpoint_type="STANDARD")
existing = [i["name"] for i in vsc.list_indexes(VS_ENDPOINT).get("vector_indexes", [])]
if VS_INDEX in existing:                       # re-run: re-embed the updated chunks
    idx = vsc.get_index(VS_ENDPOINT, VS_INDEX); idx.sync(); idx.wait_until_ready()
else:
    vsc.create_delta_sync_index_and_wait(endpoint_name=VS_ENDPOINT, index_name=VS_INDEX,
        source_table_name=f"{FQ}.contract_chunks", pipeline_type="TRIGGERED", primary_key="chunk_id",
        embedding_source_column="chunk", embedding_model_endpoint_name=EMBEDDING_MODEL)
print(vsc.get_index(VS_ENDPOINT, VS_INDEX).similarity_search(   # SUP-101 amendment should be top 3
    query_text="liquidated damages percentage ACME SUPPLIES", columns=["doc_name"], num_results=3))

spark.sql(f"""CREATE OR REPLACE FUNCTION {FQ}.search_contracts(query STRING)
RETURNS TABLE (doc_name STRING, supplier_id STRING, doc_type STRING, chunk STRING)
COMMENT 'Searches supplier MSAs and amendments: grace period, liquidated damages %, liability cap, force majeure.'
RETURN SELECT doc_name, supplier_id, doc_type, chunk FROM vector_search(index => '{VS_INDEX}', query_text => query, num_results => 5)""")

## 4. [UI] Genie Agent (data tool for the Agent Bricks agent)
1. **Genie → New → Genie Agent** `SC Risk Analyst (eval)`, serverless warehouse.
2. Add the 6 tables in `<CATALOG>.<SCHEMA>` + `mv_supply_chain*`; add documents (`<VOLUME>/contracts/` if supported, else table `<SCHEMA>.contract_docs`).
3. No instructions, SQL examples, synonyms or joins.
4. Test in the Genie UI: "How many late deliveries were there in 2025?" (expect 513) and "What is the liability cap in the WAYNE METALS agreement?" (expect USD 120,000).

Next: build the Agent Bricks agent (README.md, Step 3). It uses this Genie Agent's ID and `search_contracts` as its two tools.

## 5. Governance test (run AFTER the baseline questions; then ask Q36 as a user in `sc_emea_analyst`)

In [ ]:
EMEA_GROUP = "sc_emea_analyst"   # account group for the restricted test user (create it and add the user first)
spark.sql(f"""CREATE OR REPLACE FUNCTION {FQ}.plant_region_filter(p_plant_id STRING)
RETURN is_account_group_member('admins') OR NOT is_account_group_member('{EMEA_GROUP}')
    OR EXISTS (SELECT 1 FROM {FQ}.dim_plant p WHERE p.plant_id = p_plant_id AND p.plant_region = 'EMEA')""")
for t in ["fact_purchase_orders", "fact_downtime_events", "fact_inventory_daily"]:
    spark.sql(f"ALTER TABLE {FQ}.{t} SET ROW FILTER {FQ}.plant_region_filter ON (plant_id)")
for g in [f"USE CATALOG ON CATALOG {CATALOG}", f"USE SCHEMA, SELECT ON SCHEMA {FQ}",
          f"READ VOLUME ON VOLUME {FQ}.{VOLUME_NAME}", f"EXECUTE ON FUNCTION {FQ}.search_contracts"]:
    spark.sql(f"GRANT {g} TO `{EMEA_GROUP}`")
# undo: ALTER TABLE <catalog>.<schema>.<table> DROP ROW FILTER